In [6]:
import os
import json
import numpy as np
import torch

from datasets import load_dataset

from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification
)

from torch.optim import AdamW
from torch.utils.data import DataLoader

from tqdm.auto import tqdm

from seqeval.metrics import (
    classification_report,
    f1_score,
    precision_score,
    recall_score
)

In [7]:
print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA Available: True
GPU: Tesla T4


In [8]:
DATA_DIR = "/kaggle/input/datasets/anaya00/bc5cdr-biomedical-ner/bc5cdr"

dataset = load_dataset(
    "json",
    data_files={
        "train": os.path.join(DATA_DIR, "train.json"),
        "validation": os.path.join(DATA_DIR, "valid.json"),
        "test": os.path.join(DATA_DIR, "test.json")
    }
)

dataset

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['tags', 'tokens'],
        num_rows: 5228
    })
    validation: Dataset({
        features: ['tags', 'tokens'],
        num_rows: 5330
    })
    test: Dataset({
        features: ['tags', 'tokens'],
        num_rows: 5865
    })
})

In [9]:
with open(os.path.join(DATA_DIR, "label.json")) as f:
    label2id = json.load(f)

id2label = {
    int(v): k
    for k, v in label2id.items()
}

print("Number of labels:", len(label2id))
print(label2id)

Number of labels: 5
{'O': 0, 'B-Chemical': 1, 'B-Disease': 2, 'I-Disease': 3, 'I-Chemical': 4}


In [10]:
model_checkpoint = "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext"

tokenizer = AutoTokenizer.from_pretrained(
    model_checkpoint
)

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

In [11]:
def tokenize_and_align_labels(examples):

    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        padding="max_length",
        max_length=128,
        is_split_into_words=True
    )

    labels = []

    for i, label in enumerate(examples["tags"]):

        word_ids = tokenized_inputs.word_ids(batch_index=i)

        previous_word_idx = None

        label_ids = []

        for word_idx in word_ids:

            if word_idx is None:
                label_ids.append(-100)

            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])

            else:
                label_ids.append(-100)

            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels

    return tokenized_inputs

In [12]:
tokenized_datasets = dataset.map(
    tokenize_and_align_labels,
    batched=True
)

tokenized_datasets

Map:   0%|          | 0/5228 [00:00<?, ? examples/s]

Map:   0%|          | 0/5330 [00:00<?, ? examples/s]

Map:   0%|          | 0/5865 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['tags', 'tokens', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 5228
    })
    validation: Dataset({
        features: ['tags', 'tokens', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 5330
    })
    test: Dataset({
        features: ['tags', 'tokens', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 5865
    })
})

In [13]:
def collate_fn(batch):

    input_ids = [
        torch.tensor(ex["input_ids"])
        for ex in batch
    ]

    attention_masks = [
        torch.tensor(ex["attention_mask"])
        for ex in batch
    ]

    labels = [
        torch.tensor(ex["labels"])
        for ex in batch
    ]

    return {

        "input_ids": torch.nn.utils.rnn.pad_sequence(
            input_ids,
            batch_first=True,
            padding_value=0
        ),

        "attention_mask": torch.nn.utils.rnn.pad_sequence(
            attention_masks,
            batch_first=True,
            padding_value=0
        ),

        "labels": torch.nn.utils.rnn.pad_sequence(
            labels,
            batch_first=True,
            padding_value=-100
        )
    }


train_dataloader = DataLoader(
    tokenized_datasets["train"],
    batch_size=16,
    shuffle=True,
    collate_fn=collate_fn
)

val_dataloader = DataLoader(
    tokenized_datasets["validation"],
    batch_size=16,
    shuffle=False,
    collate_fn=collate_fn
)

test_dataloader = DataLoader(
    tokenized_datasets["test"],
    batch_size=16,
    shuffle=False,
    collate_fn=collate_fn
)

print("Dataloaders Ready")

Dataloaders Ready


In [14]:
num_labels = len(label2id)

model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model.to(device)

optimizer = AdamW(
    model.parameters(),
    lr=3e-5,
    weight_decay=0.01
)

print("Device:", device)

pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Device: cuda


In [15]:
def evaluate_ner(model, dataloader):

    model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():

        for batch in tqdm(dataloader):

            batch = {
                k: v.to(device)
                for k, v in batch.items()
            }

            outputs = model(**batch)

            preds = torch.argmax(
                outputs.logits,
                dim=-1
            )

            labels = batch["labels"]

            for pred, label in zip(preds, labels):

                true_pred = []
                true_label = []

                for p, l in zip(pred, label):

                    if l != -100:

                        true_pred.append(
                            id2label[int(p)]
                        )

                        true_label.append(
                            id2label[int(l)]
                        )

                all_preds.append(true_pred)
                all_labels.append(true_label)

    precision = precision_score(
        all_labels,
        all_preds
    )

    recall = recall_score(
        all_labels,
        all_preds
    )

    f1 = f1_score(
        all_labels,
        all_preds
    )

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

In [16]:
best_f1 = 0

MODEL_DIR = "/kaggle/working/best_model"

os.makedirs(
    MODEL_DIR,
    exist_ok=True
)

num_epochs = 3

for epoch in range(num_epochs):

    model.train()

    total_loss = 0

    progress_bar = tqdm(
        train_dataloader,
        desc=f"Epoch {epoch+1}/{num_epochs}"
    )

    for batch in progress_bar:

        batch = {
            k: v.to(device)
            for k, v in batch.items()
        }

        outputs = model(**batch)

        loss = outputs.loss

        loss.backward()

        optimizer.step()
        optimizer.zero_grad()

        total_loss += loss.item()

        progress_bar.set_postfix(
            loss=loss.item()
        )

    avg_loss = total_loss / len(train_dataloader)

    metrics = evaluate_ner(
        model,
        val_dataloader
    )

    print(
        f"\nEpoch {epoch+1}"
    )

    print(
        f"Train Loss: {avg_loss:.4f}"
    )

    print(
        f"Precision: {metrics['precision']:.4f}"
    )

    print(
        f"Recall: {metrics['recall']:.4f}"
    )

    print(
        f"F1: {metrics['f1']:.4f}"
    )

    if metrics["f1"] > best_f1:

        best_f1 = metrics["f1"]

        model.save_pretrained(
            MODEL_DIR
        )

        tokenizer.save_pretrained(
            MODEL_DIR
        )

        print(
            f"Saved Best Model (F1={best_f1:.4f})"
        )

Epoch 1/3:   0%|          | 0/327 [00:00<?, ?it/s]

  0%|          | 0/334 [00:00<?, ?it/s]


Epoch 1
Train Loss: 0.1393
Precision: 0.8645
Recall: 0.8946
F1: 0.8793


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved Best Model (F1=0.8793)


...

Epoch 3/3:   0%|          | 0/327 [00:00<?, ?it/s]

  0%|          | 0/334 [00:00<?, ?it/s]


Epoch 3
Train Loss: 0.0265
Precision: 0.8994
Recall: 0.9135
F1: 0.9064


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved Best Model (F1=0.9064)


In [17]:
best_model = AutoModelForTokenClassification.from_pretrained(
    MODEL_DIR
)

best_model.to(device)

test_metrics = evaluate_ner(
    best_model,
    test_dataloader
)

print("\nFinal Test Metrics")

print(test_metrics)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

  0%|          | 0/367 [00:00<?, ?it/s]


Final Test Metrics
{'precision': np.float64(0.8791812281577633), 'recall': np.float64(0.8978280819822576), 'f1': np.float64(0.888406820704268)}


In [18]:
!zip -r best_model.zip best_model

  adding: best_model/ (stored 0%)
  adding: best_model/config.json (deflated 54%)
  adding: best_model/tokenizer_config.json (deflated 41%)
  adding: best_model/tokenizer.json (deflated 71%)
  adding: best_model/model.safetensors (deflated 7%)
